[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vladimiracunadev-create/python-data-science-bootcamp/blob/master/classes/09-machine-learning-intro/notebook.ipynb)

# Clase 09 - Introduccion al Machine Learning
**Objetivo:** entrenar tu primer modelo predictivo con scikit-learn

**Que aprenderemos:**
- Que es el Machine Learning supervisado
- Como dividir datos en train/test
- Regresion Lineal: entrenar y evaluar
- Metricas: MAE, RMSE, R2
- Que es una linea base (baseline)


## Flujo de Trabajo de ML

```
FLUJO DE TRABAJO DE ML
======================

Datos historicos            Preprocesamiento         Entrenamiento
(ventas_tienda.csv)    -->  (seleccionar X, y)   --> model.fit(X_train, y_train)
                                   |
                         Train/Test Split                Evaluacion
                         (80% train / 20% test)  -->   model.predict(X_test)
                                                         MAE, RMSE, R2
```

> **Machine Learning Supervisado:** Le damos al modelo ejemplos con respuestas conocidas
> para que aprenda a predecir nuevas respuestas.


In [ ]:
# Qué hace: separa variables, entrena un modelo supervisado y obtiene predicciones o scores.
# Para qué sirve: permite conectar la pregunta predictiva con una primera solución medible.

# ============================================================
# BLOQUE: Importaciones
# ============================================================

# pandas: libreria para manipular tablas de datos (DataFrames)
import pandas as pd

# numpy: libreria para operaciones matematicas y arrays
import numpy as np

# matplotlib: libreria para crear graficos
import matplotlib.pyplot as plt

# sklearn.model_selection: herramientas para dividir datos
from sklearn.model_selection import train_test_split

# LinearRegression: el modelo que vamos a entrenar
from sklearn.linear_model import LinearRegression

# Metricas de evaluacion para regresion
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Confirmamos que todo se cargo bien
print("Librerias cargadas correctamente")
print(f"  pandas version: {pd.__version__}")
print(f"  scikit-learn instalado correctamente")


## Seccion 1: Cargar y Explorar los Datos

Antes de entrenar cualquier modelo, necesitamos entender nuestros datos.
Siempre hacemos EDA (Exploratory Data Analysis) primero.


In [ ]:
# Qué hace: carga el dataset base y deja la tabla lista para inspección o limpieza.
# Para qué sirve: asegura que el análisis parta desde una fuente común y verificable.

# ============================================================
# BLOQUE: Carga y Exploracion de Datos
# ============================================================

# Paso 1: Cargar el archivo CSV en un DataFrame
# pd.read_csv() lee un archivo separado por comas y lo convierte en tabla
# '../../datasets/ventas_tienda.csv' es la ruta relativa al archivo
df = pd.read_csv('../../datasets/ventas_tienda.csv')

# Paso 2: Ver las dimensiones del dataset
# .shape devuelve una tupla (filas, columnas)
print(f"Dimensiones del dataset: {df.shape}")  # Ejemplo: (500, 7)
print(f"  Filas: {df.shape[0]}")
print(f"  Columnas: {df.shape[1]}")

# Paso 3: Ver las primeras filas para entender la estructura
# .head(5) muestra las primeras 5 filas (por defecto son 5)
print("\nPrimeras 5 filas:")
df.head(5)


In [ ]:
# Qué hace: ejecuta el paso actual del ejercicio o laboratorio.
# Para qué sirve: deja explícita la intención del bloque y ayuda a revisarlo con más criterio.

# ============================================================
# BLOQUE: Informacion del Dataset
# ============================================================

# .info() muestra el tipo de dato de cada columna y si hay valores nulos
# dtype: tipo de dato (object=texto, int64=entero, float64=decimal)
print("Informacion del dataset:")
df.info()

# .describe() calcula estadisticas basicas: media, desviacion, min, max, percentiles
print("\nEstadisticas descriptivas:")
df.describe()


## Que son Features (X) y Target (y)

```
DATASET: ventas_tienda.csv
==========================
| fecha | sucursal | categoria | unidades_vendidas | precio_unitario | descuento_pct | total_venta |
                                       ^                   ^                 ^                ^
                              FEATURES (X): variables de entrada      TARGET (y): lo que queremos predecir
                              (lo que usamos para predecir)            (total_venta)

ANALOGIA:
  Si quieres predecir el precio de una casa:
  X (features) = metros_cuadrados, habitaciones, ubicacion
  y (target)   = precio_venta
```


In [ ]:
# Qué hace: ejecuta el paso actual del ejercicio o laboratorio.
# Para qué sirve: deja explícita la intención del bloque y ayuda a revisarlo con más criterio.

# ============================================================
# BLOQUE: Definir Features (X) y Target (y)
# ============================================================

# Paso 1: Seleccionar las columnas numericas que usaremos como features
# Solo usamos columnas numericas por ahora (sin fechas ni texto)
# 'unidades_vendidas': cuantas unidades se vendieron
# 'precio_unitario': precio de cada unidad
# 'descuento_pct': porcentaje de descuento aplicado
feature_columns = ['unidades_vendidas', 'precio_unitario', 'descuento_pct']

# Paso 2: Crear la matriz de features X
# df[lista_de_columnas] selecciona multiples columnas del DataFrame
# X es una matriz de forma (n_filas, n_features)
X = df[feature_columns]

# Paso 3: Crear el vector target y
# df['columna'] selecciona una sola columna (Serie)
# y es el valor que queremos predecir: total_venta
y = df['total_venta']

# Paso 4: Verificar las dimensiones
print(f"Shape de X (features): {X.shape}")   # (n_filas, n_columnas)
print(f"Shape de y (target):   {y.shape}")   # (n_filas,)
print(f"\nColumnas en X: {list(X.columns)}")
print(f"Target (y): total_venta")

# Verificar que no hay valores nulos en X ni en y
# .isnull().sum() cuenta los NaN por columna
print(f"\nValores nulos en X:")
print(X.isnull().sum())
print(f"Valores nulos en y: {y.isnull().sum()}")


## Seccion 2: Division Train/Test

```
SCHEMA: Train/Test Split
========================
Dataset completo (500 filas)
          |
    train_test_split(test_size=0.2)
          |
+------------------+  +------------+
|  X_train         |  |  X_test    |
|  y_train         |  |  y_test    |
|  (400 filas)     |  | (100 fil.) |
|  Para entrenar   |  | Para eval. |
+------------------+  +------------+

POR QUE separamos?
  Si evaluamos con los mismos datos de entrenamiento,
  el modelo 'hace trampa' - memorizo las respuestas.
  Necesitamos datos NUEVOS para saber si realmente aprendio.
```

> **Regla general:** 80% para entrenar, 20% para evaluar


In [ ]:
# Qué hace: divide los datos en entrenamiento y prueba antes de modelar.
# Para qué sirve: reserva ejemplos no vistos para evaluar si el modelo generaliza.

# ============================================================
# BLOQUE: Division Train/Test
# ============================================================

# train_test_split divide los datos en conjuntos de entrenamiento y prueba
# Parametros:
#   X, y         --> los datos a dividir (features y target)
#   test_size    --> proporcion para test (0.2 = 20%)
#   random_state --> semilla para reproducibilidad (mismo resultado siempre)
X_train, X_test, y_train, y_test = train_test_split(
    X,              # matriz de features
    y,              # vector target
    test_size=0.2,  # 20% para test, 80% para train
    random_state=42 # semilla aleatoria para reproducibilidad
)

# Verificar el tamano de cada conjunto
# Debe ser aproximadamente 80/20
print("Tamanos de los conjuntos:")
print(f"  X_train: {X_train.shape}  (filas de entrenamiento)")
print(f"  X_test:  {X_test.shape}   (filas de prueba)")
print(f"  y_train: {y_train.shape}")
print(f"  y_test:  {y_test.shape}")

# Calcular porcentajes para verificar
total = len(X)                           # total de filas originales
pct_train = len(X_train) / total * 100   # % en train
pct_test  = len(X_test)  / total * 100   # % en test
print(f"\nPorcentajes: {pct_train:.0f}% train / {pct_test:.0f}% test")


## Seccion 3: Regresion Lineal

```
FORMULA DE LA REGRESION LINEAL
==============================

  y = b0 + b1*x1 + b2*x2 + b3*x3 + ...

  Donde:
    y  = prediccion (total_venta)
    b0 = intercepto (valor base cuando todo es 0)
    b1 = coeficiente de unidades_vendidas
    b2 = coeficiente de precio_unitario
    b3 = coeficiente de descuento_pct
    x1, x2, x3 = valores de las features

  Ejemplo:
    total_venta = 10 + 5.2*unidades + 0.98*precio - 2.1*descuento

  El modelo APRENDE los coeficientes b0, b1, b2... durante .fit()
```


In [ ]:
# Qué hace: separa variables, entrena un modelo supervisado y obtiene predicciones o scores.
# Para qué sirve: permite conectar la pregunta predictiva con una primera solución medible.

# ============================================================
# BLOQUE: Crear, Entrenar y Predecir con Regresion Lineal
# ============================================================

# Paso 1: Crear una instancia del modelo
# LinearRegression() crea el modelo, pero todavia no sabe nada
# Es como contratar a un estudiante: sabe el metodo pero no el contenido
modelo = LinearRegression()

# Paso 2: Entrenar el modelo con los datos de entrenamiento
# .fit(X_train, y_train) --> el modelo aprende los coeficientes optimos
# Este paso puede tardar para datasets grandes
modelo.fit(X_train, y_train)
print("Modelo entrenado exitosamente")

# Paso 3: Ver los coeficientes que aprendio el modelo
# modelo.coef_ contiene los coeficientes b1, b2, b3...
# modelo.intercept_ contiene b0 (el termino independiente)
print(f"\nCoeficientes del modelo:")
for nombre, coef in zip(feature_columns, modelo.coef_):
    print(f"  {nombre}: {coef:.4f}")
print(f"  Intercepto (b0): {modelo.intercept_:.4f}")

# Paso 4: Hacer predicciones en datos de TEST (datos nuevos, nunca vistos)
# .predict(X_test) aplica la formula aprendida a los datos de prueba
# y_pred contiene las predicciones del modelo
y_pred = modelo.predict(X_test)

# Paso 5: Ver algunas predicciones vs valores reales
print("\nComparacion: Predicciones vs Valores Reales (primeras 5):")
print(f"{'Real':>12}  {'Prediccion':>12}  {'Diferencia':>12}")
print("-" * 40)
for real, pred in zip(y_test.values[:5], y_pred[:5]):
    diff = real - pred  # diferencia entre valor real y prediccion
    print(f"{real:>12.2f}  {pred:>12.2f}  {diff:>12.2f}")


## Seccion 4: Metricas de Evaluacion

```
TABLA DE METRICAS PARA REGRESION
=================================
| Metrica | Formula              | Interpretacion          | Mejor valor |
|---------|---------------------|-------------------------|-------------|
| MAE     | mean(|y - y_pred|)  | Error promedio en $      | Mas bajo    |
| RMSE    | sqrt(mean((y-y_p)2))| Penaliza errores grandes | Mas bajo    |
| R2      | 1 - SS_res/SS_tot   | % de varianza explicada  | Maximo = 1  |

R2 = 0.85 significa que el modelo explica el 85% de la variacion en las ventas
R2 = 1.0  significa prediccion perfecta
R2 = 0.0  el modelo es tan malo como predecir siempre la media
R2 < 0    el modelo es peor que predecir siempre la media
```


In [ ]:
# Qué hace: ejecuta el paso actual del ejercicio o laboratorio.
# Para qué sirve: deja explícita la intención del bloque y ayuda a revisarlo con más criterio.

# ============================================================
# BLOQUE: Calcular Metricas de Evaluacion
# ============================================================

# TODO: Calcula las 3 metricas usando y_test e y_pred

# MAE (Mean Absolute Error = Error Absoluto Medio)
# Promedio de los errores absolutos |real - prediccion|
mae = ___  # mean_absolute_error(y_test, y_pred)

# MSE (Mean Squared Error = Error Cuadratico Medio)
# Promedio de los errores al cuadrado (penaliza errores grandes)
mse = ___  # mean_squared_error(y_test, y_pred)

# RMSE (Root Mean Squared Error = Raiz del Error Cuadratico Medio)
# Raiz cuadrada del MSE, en las mismas unidades que y
rmse = ___  # np.sqrt(mse)

# R2 Score (Coeficiente de Determinacion)
# Que tanto del target explica el modelo (0 a 1, mas alto es mejor)
r2 = ___  # r2_score(y_test, y_pred)

# Mostrar resultados formateados
print("=" * 40)
print("METRICAS DE EVALUACION DEL MODELO")
print("=" * 40)
print(f"  MAE  (Error Absoluto Medio):  {mae:.2f}")
print(f"  RMSE (Raiz Error Cuadratico): {rmse:.2f}")
print(f"  R2   (Coef. Determinacion):   {r2:.4f}")
print("=" * 40)

# Interpretacion del R2
print(f"\nInterpretacion: el modelo explica el {r2*100:.1f}% de la variabilidad en total_venta")


In [ ]:
# Qué hace: ejecuta el paso actual del ejercicio o laboratorio.
# Para qué sirve: deja explícita la intención del bloque y ayuda a revisarlo con más criterio.

# ============================================================
# BLOQUE: Baseline (Linea Base)
# ============================================================
# La BASELINE es el modelo mas simple posible
# Para regresion: predecir siempre la media del target
# Si nuestro modelo no supera la baseline, no sirve de nada

# Paso 1: Calcular la media de y_train (lo que conocemos en entrenamiento)
# Usamos y_train (no y_test) para no 'hacer trampa'
media_entrenamiento = y_train.mean()  # media de ventas en train
print(f"Media del target en train: {media_entrenamiento:.2f}")

# Paso 2: Crear predicciones baseline: siempre predecir la media
# np.full(n, valor) crea un array de n elementos con el valor dado
y_pred_baseline = np.full(len(y_test), media_entrenamiento)
print(f"Baseline predice siempre: {media_entrenamiento:.2f}")

# Paso 3: Calcular metricas de la baseline
mae_baseline  = mean_absolute_error(y_test, y_pred_baseline)
rmse_baseline = np.sqrt(mean_squared_error(y_test, y_pred_baseline))
r2_baseline   = r2_score(y_test, y_pred_baseline)  # deberia ser ~0

# Paso 4: Comparar modelo vs baseline
print("\nCOMPARACION: Modelo vs Baseline")
print(f"{'Metrica':<8}  {'Nuestro Modelo':>16}  {'Baseline':>12}  {'Mejora':>10}")
print("-" * 52)
print(f"{'MAE':<8}  {mae:>16.2f}  {mae_baseline:>12.2f}  {(1-mae/mae_baseline)*100:>9.1f}%")
print(f"{'RMSE':<8}  {rmse:>16.2f}  {rmse_baseline:>12.2f}  {(1-rmse/rmse_baseline)*100:>9.1f}%")
print(f"{'R2':<8}  {r2:>16.4f}  {r2_baseline:>12.4f}")


## Seccion 5: Visualizacion de Resultados

Los graficos nos ayudan a entender si el modelo esta funcionando bien.
Un buen modelo tendra puntos cercanos a la linea diagonal en el grafico de predicciones.


In [ ]:
# Qué hace: construye una visualización a partir del resumen preparado.
# Para qué sirve: permite comunicar el hallazgo central con una lectura más rápida.

# ============================================================
# BLOQUE: Grafico Predicciones vs Valores Reales
# ============================================================

# Crear figura con un solo grafico
# figsize=(8, 6) define el tamano en pulgadas (ancho, alto)
fig, ax = plt.subplots(figsize=(8, 6))

# Scatter plot: cada punto es una observacion
# x = valor real, y = valor predicho
# alpha=0.5 hace los puntos semitransparentes (util cuando hay muchos)
ax.scatter(
    y_test,      # eje x: valores reales
    y_pred,      # eje y: predicciones del modelo
    alpha=0.5,   # transparencia (0=invisible, 1=opaco)
    color='steelblue',  # color de los puntos
    edgecolors='white', # borde blanco para mejor visualizacion
    s=60,        # tamano de cada punto
    label='Predicciones'
)

# Linea diagonal perfecta (y = x)
# Si el modelo fuera perfecto, todos los puntos estarian sobre esta linea
# np.linspace(min, max, n) crea n puntos uniformemente espaciados
min_val = min(y_test.min(), y_pred.min())  # valor minimo entre reales y predichos
max_val = max(y_test.max(), y_pred.max())  # valor maximo entre reales y predichos
linea_perfecta = np.linspace(min_val, max_val, 100)

# Dibujar la linea perfecta en rojo punteado
ax.plot(
    linea_perfecta,  # valores en x
    linea_perfecta,  # valores en y (mismos, por eso es diagonal)
    'r--',           # 'r' = rojo, '--' = linea punteada
    linewidth=2,     # grosor de la linea
    label='Prediccion perfecta'
)

# Etiquetas y titulo
ax.set_xlabel('Valores Reales (total_venta)', fontsize=12)    # eje x
ax.set_ylabel('Valores Predichos', fontsize=12)               # eje y
ax.set_title(f'Predicciones vs Valores Reales\nR² = {r2:.4f}', fontsize=14)
ax.legend(fontsize=11)   # mostrar leyenda
ax.grid(True, alpha=0.3) # cuadricula con transparencia

# Ajustar el layout y mostrar
plt.tight_layout()  # ajusta automaticamente los margenes
plt.show()


In [ ]:
# Qué hace: construye una visualización a partir del resumen preparado.
# Para qué sirve: permite comunicar el hallazgo central con una lectura más rápida.

# ============================================================
# BLOQUE: Histograma de Residuos
# ============================================================
# RESIDUO = valor real - valor predicho
# En un buen modelo, los residuos deben:
#   1. Estar centrados en 0 (sin sesgo sistematico)
#   2. Tener distribucion aproximadamente normal (campana de Gauss)

# Calcular los residuos
# numpy resta elemento a elemento: real - prediccion
residuos = y_test.values - y_pred  # array de diferencias

# Crear figura
fig, ax = plt.subplots(figsize=(8, 5))

# Histograma de residuos
# bins=40 divide el rango en 40 barras
# edgecolor='white' dibuja borde blanco entre barras
ax.hist(
    residuos,       # datos a graficar
    bins=40,        # numero de barras
    color='steelblue',   # color de relleno
    edgecolor='white',   # color del borde de cada barra
    alpha=0.8            # transparencia
)

# Linea vertical en x=0 para ver si los residuos estan centrados
ax.axvline(
    x=0,            # posicion en el eje x
    color='red',    # color rojo para destacar
    linewidth=2,    # grosor
    linestyle='--', # punteada
    label='Residuo = 0 (perfecto)'
)

# Etiquetas
ax.set_xlabel('Residuo (Real - Prediccion)', fontsize=12)
ax.set_ylabel('Frecuencia', fontsize=12)
ax.set_title('Distribucion de Residuos\n(Debe ser campana centrada en 0)', fontsize=14)
ax.legend()
ax.grid(True, alpha=0.3)

# Estadisticas de residuos
print(f"Media de residuos: {residuos.mean():.4f} (idealmente cerca de 0)")
print(f"Std  de residuos:  {residuos.std():.4f}")

plt.tight_layout()
plt.show()


## Resumen de la Clase 09

### Checklist de lo aprendido:

- [ ] Cargar datos y hacer EDA basico
- [ ] Definir features (X) y target (y)
- [ ] Dividir datos en train/test con `train_test_split`
- [ ] Entrenar un modelo `LinearRegression` con `.fit()`
- [ ] Hacer predicciones con `.predict()`
- [ ] Calcular MAE, RMSE, R2
- [ ] Comparar con una linea base
- [ ] Visualizar predicciones y residuos

### Conceptos clave:
- **Overfitting:** modelo muy ajustado a train, falla en test
- **Underfitting:** modelo demasiado simple, falla en ambos
- **R2 alto (> 0.8):** buen modelo
- **R2 bajo (< 0.5):** necesita mejoras

### Proxima clase:
Clase 10 - Modelos de Clasificacion: Decision Trees y Logistic Regression
